Odnajdujemy projekt i umożliwiamy import wspólnych modułów.

In [ ]:
from pathlib import Path
import sys

project_root = next(
    (
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "src/restore_pipeline/super_resolution").is_dir()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError("Uruchom notebook wewnątrz katalogu projektu.")

source_path = str(project_root / "src")

if source_path not in sys.path:
    sys.path.insert(0, source_path)


Importujemy wspólną ocenę eksperymentów i podgląd zapisanych raportów.

In [ ]:
import torch
from IPython.display import Image as DisplayImage, display
from restore_pipeline.super_resolution.evaluation import benchmark_experiments

from restore_pipeline.super_resolution.paths import display_path


Wybieramy katalogi eksperymentów do pełnego testu, wspólny batch size i ograniczony zestaw przykładów.

In [ ]:
DATASET_PATH = project_root / "notebooks/igoh/data/partitioned_dataset"
EXPERIMENTS_PATH = project_root / "notebooks/igoh/models_weights/experiments"
RESULTS_PATH = project_root / "notebooks/igoh/data/sr_results"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

RUN_PATHS = []
BATCH_SIZE = 64
NUM_WORKERS = 2
PREVIEW_INDICES = [5, 6, 7, 8, 9]
CASE_COUNT = 3
SHOW_PROGRESS = True


Oceniamy najlepsze checkpointy na obrazach pełnego testu, mierząc forward po rozgrzewce i licząc bicubic tylko raz.

In [ ]:
summary = benchmark_experiments(
    RUN_PATHS,
    DATASET_PATH,
    RESULTS_PATH,
    DEVICE,
    split="test",
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    preview_indices=PREVIEW_INDICES,
    case_count=CASE_COUNT,
    show_progress=SHOW_PROGRESS,
)

display(summary.round(5))


Porównujemy wspólne metryki oraz czas samego forward bez transferów i obliczania metryk.

In [ ]:
display(DisplayImage(filename=str(RESULTS_PATH / "test_comparison.png")))
display(DisplayImage(filename=str(RESULTS_PATH / "test_inference_time.png")))


Oglądamy te same fragmenty i najsłabsze przypadki, a pozostałe grafiki znajdują się w katalogach raportów.

In [ ]:
for run_path in RUN_PATHS:
    report_path = run_path / "test"
    print(display_path(report_path))
    display(DisplayImage(filename=str(report_path / "details.png")))
    display(DisplayImage(filename=str(report_path / "worst_cases.png")))
